# EEEM068: Industrial Waste Classification
## Notebook 02 — Data Pipeline

**Author:** Scott Lewis  
**Date:** March 2026  

---

### Objectives

This notebook demonstrates and validates the full data pipeline:

1. CSV-based dataset loader
2. Preprocessing and augmentation transforms
3. DataLoader configuration
4. Augmentation visualisation — showing what the model actually sees
5. Class weight computation for imbalance handling

---

## 0. Setup & Imports

In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('../../')
DATASET_DIR  = Path('dataset')
DATA_ROOT    = PROJECT_ROOT / 'data'
MAPPING_JSON = DATASET_DIR / 'class_mapping.json'

with open(MAPPING_JSON) as f:
    mapping = json.load(f)
label_to_class = {int(k): v for k, v in mapping['label_to_class'].items()}
n_classes      = mapping['n_classes']

print('Setup complete')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

Setup complete
Device: cuda


## 1. Custom CSV Dataset

In [2]:
class WaRPDataset(Dataset):
    """
    CSV-based PyTorch Dataset for WaRP-C.

    Reads image paths and labels from a CSV file generated by prepare_data.py.
    CSV format: filepath, label, class_name

    Args:
        csv_path:     path to train.csv / val.csv / test.csv
        project_root: project root directory (filepaths in CSV are relative to this)
        transform:    torchvision transforms to apply
    """

    def __init__(self, csv_path: str, project_root: str, transform=None):
        self.df           = pd.read_csv(csv_path)
        self.project_root = Path(project_root)
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = self.project_root / row['filepath']
        label     = int(row['label'])

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label

    def get_class_weights(self) -> torch.Tensor:
        """Compute inverse frequency class weights for weighted loss."""
        counts  = self.df['label'].value_counts().sort_index()
        weights = 1.0 / (counts.values + 1e-6)
        return torch.FloatTensor(weights)


print('WaRPDataset class defined')

WaRPDataset class defined


In [3]:
# test dataset loading without transforms
raw_dataset = WaRPDataset(
    csv_path     = DATASET_DIR / 'train.csv',
    project_root = PROJECT_ROOT,
    transform    = None
)

print(f'Train dataset size: {len(raw_dataset)}')

# test a single item
img, label = raw_dataset[0]
print(f'Sample image type:  {type(img)}')
print(f'Sample image size:  {img.size}')
print(f'Sample label:       {label} ({label_to_class[label]})')

Train dataset size: 7069


FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\Data\\train\\bottle-blue\\Monitoring_photo1_04-Mar_03-09-16_01.jpg'

## 2. Transforms

In [ ]:
# ImageNet normalisation — required for all pretrained backbones
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

IMG_SIZE = (224, 224)

# Training transforms — includes augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1          # kept low — hue shifts mislead waste classifiers
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation/test transforms — no augmentation, deterministic
val_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('Transforms defined')
print(f'\nTrain transforms:\n{train_transforms}')
print(f'\nVal transforms:\n{val_transforms}')

## 3. Augmentation Visualisation

Showing what the model actually sees during training — the same image with different augmentations applied.

In [ ]:
def denormalise(tensor: torch.Tensor) -> np.ndarray:
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    img  = tensor * std + mean
    img  = img.permute(1,2,0).numpy()
    return np.clip(img, 0, 1)


# pick one image and apply train transforms multiple times
img_raw, label = raw_dataset[0]
class_name     = label_to_class[label]

n_aug = 8
fig, axes = plt.subplots(2, n_aug // 2, figsize=(16, 7))
axes = axes.flatten()

# first show original
axes[0].imshow(img_raw.resize(IMG_SIZE))
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

# then show augmented versions
for i in range(1, n_aug):
    aug_tensor = train_transforms(img_raw)
    axes[i].imshow(denormalise(aug_tensor))
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')

fig.suptitle(f'Augmentation samples — class: {class_name}', fontsize=12)
plt.tight_layout()
plt.savefig('augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: augmentation_samples.png')

## 4. DataLoaders

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0 if os.name == 'nt' else 4  # 0 on Windows, 4 on Linux/Mac
PIN_MEMORY  = torch.cuda.is_available()

train_dataset = WaRPDataset(DATASET_DIR / 'train.csv', PROJECT_ROOT, train_transforms)
val_dataset   = WaRPDataset(DATASET_DIR / 'val.csv',   PROJECT_ROOT, val_transforms)
test_dataset  = WaRPDataset(DATASET_DIR / 'test.csv',  PROJECT_ROOT, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

# verify a batch loads correctly
images, labels = next(iter(train_loader))
print(f'\nBatch image shape: {images.shape}')   # [batch, 3, 224, 224]
print(f'Batch label shape: {labels.shape}')     # [batch]
print(f'Label range: {labels.min().item()} – {labels.max().item()}')

## 5. Class Weights

In [ ]:
class_weights = train_dataset.get_class_weights()

print(f'Class weights shape: {class_weights.shape}')
print(f'Min weight: {class_weights.min():.4f} (most common class)')
print(f'Max weight: {class_weights.max():.4f} (rarest class)')
print(f'\nTop 5 highest weighted (most underrepresented) classes:')

top5_idx = class_weights.argsort(descending=True)[:5]
for idx in top5_idx:
    print(f'  [{idx:>2}] {label_to_class[idx.item()]:<35} weight: {class_weights[idx]:.4f}')

## 6. Summary

The data pipeline is validated and ready for training:

- `WaRPDataset` loads images from CSV — no dependency on folder structure
- Train transforms apply appropriate augmentation (crop, flip, colour jitter)
- Val/test transforms are deterministic (resize + centre crop only)
- Class weights computed for weighted CrossEntropyLoss
- DataLoaders configured correctly for Windows and Linux/Mac

See `03_resnet50_training.ipynb` for model training.